# 🏃 Personal Running Coach Agent Swarm
## Phase 1 — Environment & Data Foundations

**Project:** Agentic AI Course — Will Sutherland, April 2026

### Data ingestion strategy

| Source | Role | Data | Cadence |
|---|---|---|---|
| **`garmin.db`** | Primary — rich physiological data | HRV, sleep, body battery, stress, training readiness, activities, VO2max, race predictions, resting HR | Sync locally via `garmin-givemydata`, copy updated db to Drive |
| **Strava API** | Activity trigger + suffer score | Pace, distance, HR, suffer score | Every run (automated) |

> **`garmin.db` setup:** `garmin-givemydata` runs locally on your Windows machine using a
> headless Chrome browser to bypass Garmin's Cloudflare protection (which blocks all
> Python HTTP clients as of March 2026). The resulting SQLite database is copied to
> `Google Drive / running_coach / data / raw / garmin / garmin.db` where Colab reads it.
> To refresh: run `garmin-givemydata --days 7` locally, overwrite the Drive file.

### This notebook covers
1. Dependency installation
2. Secret / credential configuration
3. Mount Google Drive & initialize folder structure
4. `config.py` — shared constants
5. Garmin DB — connect, inspect tables, verify data
6. Strava API — OAuth setup & historical pull
7. Normalize Strava to shared WorkoutRecord schema
8. Enrich WorkoutRecords from Garmin DB
9. Smoke tests

---
### Folder structure
```
Google Drive / running_coach /
├── data/
│   ├── raw/
│   │   ├── garmin/       # garmin.db — synced locally, copied here
│   │   └── strava/       # Strava activity JSON pulls
│   └── processed/        # Normalized WorkoutRecord JSON (shared schema)
├── memory/
│   └── chroma/           # Persisted ChromaDB vector store
├── agents/               # One .py file per agent (Phase 4)
├── tools/                # Tool .py files (Phase 2)
├── config.py             # Shared constants
└── notebooks/            # This file lives here
```

---
## 1. Install Dependencies

In [1]:
# Core LLM + agent framework
%pip install -q google-generativeai
%pip install -q langchain langchain-google-genai langchain-community
%pip install -q langgraph

# Memory / RAG
%pip install -q chromadb

# Strava API
%pip install -q stravalib

# Data handling
%pip install -q pandas numpy

# Utilities
%pip install -q requests

# sqlite3 is part of Python stdlib — no install needed for garmin.db access

print("✅ All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

---
## 2. Configure Secrets

In Colab, open the **🔑 Secrets** panel (left sidebar, key icon) and add these:

| Secret name | Value |
|---|---|
| `GEMINI_API_KEY` | Your Google AI Studio API key |
| `STRAVA_CLIENT_ID` | From [strava.com/settings/api](https://www.strava.com/settings/api) |
| `STRAVA_CLIENT_SECRET` | From Strava API settings |
| `STRAVA_REFRESH_TOKEN` | Generated during the OAuth flow in Section 6 below |

> **No Garmin credentials needed in Colab.** Authentication happens locally on your
> Windows machine via `garmin-givemydata`. Colab only reads the resulting `garmin.db`
> file from Google Drive — no login required.

In [2]:
from google.colab import userdata

GEMINI_API_KEY        = userdata.get('key')
STRAVA_CLIENT_ID      = userdata.get('STRAVA_CLIENT_ID')
STRAVA_CLIENT_SECRET  = userdata.get('STRAVA_CLIENT_SECRET')
STRAVA_REFRESH_TOKEN  = userdata.get('STRAVA_REFRESH_TOKEN')

secrets = {
    'GEMINI_API_KEY':       GEMINI_API_KEY,
    'STRAVA_CLIENT_ID':     STRAVA_CLIENT_ID,
    'STRAVA_CLIENT_SECRET': STRAVA_CLIENT_SECRET,
    'STRAVA_REFRESH_TOKEN': STRAVA_REFRESH_TOKEN,
}
for k, v in secrets.items():
    status = '✅ loaded' if v else '❌ MISSING — add to Secrets panel'
    print(f"{k}: {status}")

GEMINI_API_KEY: ✅ loaded
STRAVA_CLIENT_ID: ✅ loaded
STRAVA_CLIENT_SECRET: ✅ loaded
STRAVA_REFRESH_TOKEN: ✅ loaded


---
## 3. Mount Google Drive & Initialize Folder Structure

Mounting Google Drive means all project files — data, ChromaDB memory, processed
workouts, agent and tool files — **persist between Colab sessions**.

When you run the cell below, Colab opens a popup asking you to sign in to Google
and grant Drive access. After authorizing, your Drive is mounted at `/content/drive/MyDrive/`.

> **Safe to re-run.** If Drive is already mounted it skips the auth popup.
> Folder creation uses `exist_ok=True` so re-running never overwrites existing data.

In [3]:
import os
from google.colab import drive

# ── Mount Google Drive ─────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted at /content/drive/MyDrive/")

# ── Base directory ─────────────────────────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/running_coach"

# ── Create folder structure ────────────────────────────────────────────────────
folders = [
    "data/raw/garmin",
    "data/raw/strava",
    "data/processed",
    "memory/chroma",
    "agents",
    "tools",
    "notebooks",
]

for folder in folders:
    path = os.path.join(BASE_DIR, folder)
    os.makedirs(path, exist_ok=True)
    print(f"📁 {path}")

# ── Key paths used throughout the notebook ────────────────────────────────────
GARMIN_DB_PATH  = os.path.join(BASE_DIR, "data/raw/garmin/garmin.db")
STRAVA_RAW_DIR  = os.path.join(BASE_DIR, "data/raw/strava")
PROC_DATA_DIR   = os.path.join(BASE_DIR, "data/processed")
CHROMA_DIR      = os.path.join(BASE_DIR, "memory/chroma")

print(f"\n✅ Folder structure ready.")
print(f"   Project root:  {BASE_DIR}")
print(f"   Garmin DB:     {GARMIN_DB_PATH}")
print(f"   DB exists:     {os.path.exists(GARMIN_DB_PATH)}")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive/MyDrive/
📁 /content/drive/MyDrive/running_coach/data/raw/garmin
📁 /content/drive/MyDrive/running_coach/data/raw/strava
📁 /content/drive/MyDrive/running_coach/data/processed
📁 /content/drive/MyDrive/running_coach/memory/chroma
📁 /content/drive/MyDrive/running_coach/agents
📁 /content/drive/MyDrive/running_coach/tools
📁 /content/drive/MyDrive/running_coach/notebooks

✅ Folder structure ready.
   Project root:  /content/drive/MyDrive/running_coach
   Garmin DB:     /content/drive/MyDrive/running_coach/data/raw/garmin/garmin.db
   DB exists:     True


---
## 4. Write `config.py` — Shared Constants

**Edit before running:** update `HR_MAX`, `RACE_DATE`, and `CURRENT_WEEKLY_KM`
to your actual values. Pace zone boundaries should match your coach-provided thresholds.

In [4]:
config_content = '''
# running_coach/config.py
# Shared constants used across all agents and tools.
# Edit to match your actual athlete profile before running Phase 2+.

# ── Athlete profile ────────────────────────────────────────────────────────────
ATHLETE_NAME        = "Will Sutherland"
TARGET_RACE         = "Fredericton Half Marathon"
RACE_DATE           = "2026-05-10"   # TODO: confirm actual race date
CURRENT_WEEKLY_KM   = 70             # TODO: update with current weekly volume

# ── Training paces (min/km) — +/- 5 seconds tolerance applied in tools ───────
# Easy run is effort-based (by feel) — no strict target, ceiling at 5:30/km
TRAINING_PACES = {
    "easy":       None,            # By feel — no target
    "marathon":   4 + 14/60,       # 4:14/km
    "threshold":  4 + 01/60,       # 4:01/km
    "1hr":        3 + 56/60,       # 3:56/km
    "fartlek":    3 + 49/60,       # 3:49/km
    "8k":         3 + 46/60,       # 3:46/km
    "vo2max":     3 + 40/60,       # 3:40/km
}
EASY_PACE_CEILING  = 4 + 50/60   # Faster than 4:50/km = too hard for easy run
PACE_TOLERANCE_SEC = 5            # +/- 5 seconds counts as on-target

# ── Heart rate zones (bpm) ────────────────────────────────────────────────────
HR_MAX = 195  # TODO: update with your measured max HR
HR_ZONES = {
    "Z1": (0,                    int(HR_MAX * 0.60)),
    "Z2": (int(HR_MAX * 0.60),   int(HR_MAX * 0.70)),
    "Z3": (int(HR_MAX * 0.70),   int(HR_MAX * 0.80)),
    "Z4": (int(HR_MAX * 0.80),   int(HR_MAX * 0.90)),
    "Z5": (int(HR_MAX * 0.90),   HR_MAX),
}

# ── Training load thresholds ───────────────────────────────────────────────────
TSB_ALERT_THRESHOLD  = -30   # [RECOVERY ALERT] when TSB drops below this
MILEAGE_RULE_PCT     = 0.10  # 10% weekly mileage increase cap
ATL_DECAY            = 7     # Acute Training Load decay constant (days)
CTL_DECAY            = 42    # Chronic Training Load decay constant (days)

# ── Data ingestion ─────────────────────────────────────────────────────────────
# Garmin DB: sync locally via garmin-givemydata, copy garmin.db to Drive
# Strava:    automated pull via OAuth on each session
STRAVA_HISTORY_DAYS  = 90

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR        = "/content/drive/MyDrive/running_coach"
GARMIN_DB_PATH  = f"{BASE_DIR}/data/raw/garmin/garmin.db"
STRAVA_RAW_DIR  = f"{BASE_DIR}/data/raw/strava"
PROC_DATA_DIR   = f"{BASE_DIR}/data/processed"
CHROMA_DIR      = f"{BASE_DIR}/memory/chroma"
AGENTS_DIR      = f"{BASE_DIR}/agents"
TOOLS_DIR       = f"{BASE_DIR}/tools"

# ── Model ──────────────────────────────────────────────────────────────────────
GEMINI_MODEL    = "gemini-2.5-flash"
EMBED_MODEL     = "models/embedding-001"
'''

config_path = os.path.join(BASE_DIR, "config.py")
with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"✅ config.py written to {config_path}")
print("   Remember to update HR_MAX, RACE_DATE, and CURRENT_WEEKLY_KM before Phase 2.")

✅ config.py written to /content/drive/MyDrive/running_coach/config.py
   Remember to update HR_MAX, RACE_DATE, and CURRENT_WEEKLY_KM before Phase 2.


---
## 5. Garmin DB — Connect, Inspect & Verify

**Before running:** make sure `garmin.db` has been copied to:
`Google Drive / running_coach / data / raw / garmin / garmin.db`

**To refresh the database locally (Windows terminal):**
```
garmin-givemydata --days 7
```
Then overwrite the file in Drive with the updated copy from:
`C:\Users\wills\.garmin-givemydata\garmin.db`

In [5]:
# ── Connect to garmin.db and inspect available tables ─────────────────────────
import sqlite3
import pandas as pd

if not os.path.exists(GARMIN_DB_PATH):
    raise FileNotFoundError(
        f"garmin.db not found at {GARMIN_DB_PATH}\n"
        "Copy it from C:\\Users\\wills\\.garmin-givemydata\\garmin.db to Drive first."
    )

conn = sqlite3.connect(GARMIN_DB_PATH)
print(f"✅ Connected to garmin.db")
print(f"   Path: {GARMIN_DB_PATH}\n")

# Show all tables and row counts
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print(f"{'Table':<40} {'Rows':>8}")
print("-" * 50)
for table in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0]['n']
    marker = " ⭐" if count > 0 else ""
    print(f"{table:<40} {count:>8,}{marker}")

print(f"\nTotal tables: {len(tables)}")

✅ Connected to garmin.db
   Path: /content/drive/MyDrive/running_coach/data/raw/garmin/garmin.db

Table                                        Rows
--------------------------------------------------
activity                                      829 ⭐
activity_exercise_sets                        837 ⭐
activity_hr_zones                             829 ⭐
activity_splits                             9,417 ⭐
activity_trends                                 0
activity_types                                152 ⭐
activity_weather                              829 ⭐
blood_pressure                                  0
body_battery                                3,673 ⭐
calories                                      873 ⭐
challenges                                      0
daily_events                                  772 ⭐
daily_movement                              3,673 ⭐
daily_summary                               3,686 ⭐
device                                          1 ⭐
earned_badges              

In [6]:
# ── Preview the key tables for coaching use ───────────────────────────────────
# These are the tables the Recovery, Planner, and Feedback agents will query.

KEY_TABLES = {
    'activities':          'Running activities with pace, HR, training load',
    'hrv':                 'HRV status, baseline, BALANCED/UNBALANCED streak',
    'sleep':               'Sleep stages, deep/REM %, SpO2, sleep score',
    'body_battery':        'Charge/drain patterns, wake value',
    'stress':              'Daily stress time-in-zone breakdown',
    'training_readiness':  'Readiness score + factor breakdown',
    'resting_heart_rate':  'Daily resting HR with 7-day rolling average',
    'training_load':       'CTL/ATL/TSB periodization metrics',
    'race_predictions':    'Predicted 5K/10K/HM/marathon times',
    'vo2max':              'VO2max trend over time',
}

print("Key table availability for your device (Forerunner 265):\n")
print(f"{'Table':<25} {'Rows':>6}  {'Role'}")
print("-" * 70)

available_tables = tables['name'].tolist()
for table, role in KEY_TABLES.items():
    if table in available_tables:
        count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0]['n']
        status = f"{count:>6,}"
    else:
        status = "   N/A"
    print(f"{table:<25} {status}  {role}")

Key table availability for your device (Forerunner 265):

Table                       Rows  Role
----------------------------------------------------------------------
activities                   N/A  Running activities with pace, HR, training load
hrv                          864  HRV status, baseline, BALANCED/UNBALANCED streak
sleep                        838  Sleep stages, deep/REM %, SpO2, sleep score
body_battery               3,673  Charge/drain patterns, wake value
stress                     3,673  Daily stress time-in-zone breakdown
training_readiness           872  Readiness score + factor breakdown
resting_heart_rate           N/A  Daily resting HR with 7-day rolling average
training_load                N/A  CTL/ATL/TSB periodization metrics
race_predictions             867  Predicted 5K/10K/HM/marathon times
vo2max                         0  VO2max trend over time


In [7]:
# ── Sample recent data from each key table ────────────────────────────────────
# Confirms data looks sensible before we build anything on top of it.

print("=== Recent Running Activities (last 5) ===")
try:
    df_act = pd.read_sql("""
        SELECT start_time_local, activity_name, activity_type,
               ROUND(distance_meters / 1000.0, 2) AS distance_km,
               ROUND(duration_seconds / 60.0, 1)  AS duration_min,
               average_hr, training_load,
               aerobic_training_effect, anaerobic_training_effect
        FROM activity
        WHERE LOWER(activity_type) LIKE '%run%'
        ORDER BY start_time_local DESC
        LIMIT 5
    """, conn)
    display(df_act)
except Exception as e:
    print(f"  ⚠️  {e}")

print("\n=== Recent HRV (last 7 days) ===")
try:
    df_hrv = pd.read_sql("""
        SELECT calendar_date, weekly_avg, last_night, status,
               baseline_low, baseline_upper
        FROM hrv
        ORDER BY calendar_date DESC
        LIMIT 7
    """, conn)
    display(df_hrv)
except Exception as e:
    print(f"  ⚠️  {e}")

print("\n=== Recent Sleep (last 5 nights) ===")
try:
    df_sleep = pd.read_sql("""
        SELECT calendar_date,
               ROUND(sleep_time_seconds / 3600.0, 2) AS total_sleep_hrs,
               ROUND(deep_sleep_seconds  / 3600.0, 2) AS deep_hrs,
               ROUND(rem_sleep_seconds   / 3600.0, 2) AS rem_hrs,
               average_spo2, avg_sleep_stress, sleep_score_feedback
        FROM sleep
        ORDER BY calendar_date DESC
        LIMIT 5
    """, conn)
    display(df_sleep)
except Exception as e:
    print(f"  ⚠️  {e}")

print("\n=== Recent Body Battery (last 7 days) ===")
try:
    df_bb = pd.read_sql("""
        SELECT calendar_date, charged, drained, highest, lowest, at_wake
        FROM body_battery
        ORDER BY calendar_date DESC
        LIMIT 7
    """, conn)
    display(df_bb)
except Exception as e:
    print(f"  ⚠️  {e}")

print("\n=== Race Predictions (most recent) ===")
try:
    df_race = pd.read_sql("""
        SELECT calendar_date, time_5k, time_10k,
               time_half_marathon, time_marathon
        FROM race_predictions
        ORDER BY calendar_date DESC
        LIMIT 3
    """, conn)
    display(df_race)
except Exception as e:
    print(f"  ⚠️  {e}")

print("\n=== Training Readiness (last 7 days) ===")
try:
    df_tr = pd.read_sql("""
        SELECT calendar_date, score, level, feedback_short,
               recovery_time, hrv_factor_feedback, sleep_history_factor_feedback
        FROM training_readiness
        ORDER BY calendar_date DESC
        LIMIT 7
    """, conn)
    display(df_tr)
except Exception as e:
    print(f"  ⚠️  {e}")

=== Recent Running Activities (last 5) ===


,start_time_local,activity_name,activity_type,distance_km,duration_min,average_hr,training_load,aerobic_training_effect,anaerobic_training_effect
0,2026-05-17 12:04:50,Halifax Running,running,1.30,7.5,129.0,13.455627,1.0,0.0
1,2026-05-17 10:55:54,Halifax Running,running,11.15,47.5,170.0,226.427383,4.5,1.0
2,2026-05-17 10:11:00,Halifax Running,running,3.74,16.0,156.0,72.683136,2.8,0.5
3,2026-05-17 08:19:33,Halifax Running,running,2.52,13.1,134.0,26.876877,2.0,0.0
4,2026-05-17 08:00:02,Halifax Running,running,3.49,13.0,166.0,85.400040,3.0,0.8



=== Recent HRV (last 7 days) ===


,calendar_date,weekly_avg,last_night,status,baseline_low,baseline_upper
0,2026-05-18,82.0,None,LOW,84.0,120.0
1,2026-05-17,82.0,None,LOW,84.0,121.0
2,2026-05-16,83.0,None,LOW,84.0,121.0
3,2026-05-15,89.0,None,UNBALANCED,85.0,120.0
4,2026-05-14,92.0,None,BALANCED,85.0,120.0
5,2026-05-13,98.0,None,BALANCED,85.0,120.0
6,2026-05-12,96.0,None,BALANCED,85.0,120.0



=== Recent Sleep (last 5 nights) ===


...

,calendar_date,charged,drained,highest,lowest,at_wake
0,2026-05-18,None,None,None,None,None
1,2026-05-17,None,None,None,None,None
2,2026-05-16,None,None,None,None,None
3,2026-05-15,None,None,None,None,None
4,2026-05-14,None,None,None,None,None
5,2026-05-13,None,None,None,None,None
6,2026-05-12,None,None,None,None,None



=== Race Predictions (most recent) ===


,calendar_date,time_5k,time_10k,time_half_marathon,time_marathon
0,2026-05-18,1055.0,2245.0,4948.0,10773.0
1,2026-05-17,1055.0,2245.0,4949.0,10775.0
2,2026-05-16,1064.0,2262.0,4984.0,10816.0



=== Training Readiness (last 7 days) ===


,calendar_date,score,level,feedback_short,recovery_time,hrv_factor_feedback,sleep_history_factor_feedback
0,2026-05-18,1.0,POOR,FOCUS_ON_RECOVERY,3711.0,POOR,MODERATE
1,2026-05-17,54.0,MODERATE,LISTEN_TO_YOUR_BODY,1166.0,MODERATE,MODERATE
2,2026-05-16,83.0,HIGH,WELL_RECOVERED,0.0,GOOD,MODERATE
3,2026-05-15,76.0,HIGH,LISTEN_TO_YOUR_BODY,0.0,MODERATE,GOOD
4,2026-05-14,74.0,MODERATE,GOOD_SLEEP_HISTORY,1.0,GOOD,GOOD
5,2026-05-13,75.0,HIGH,WELL_RECOVERED,30.0,GOOD,GOOD
6,2026-05-12,57.0,MODERATE,RECOVERY_IN_PROGRESS,1709.0,GOOD,MODERATE


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(f"{BASE_DIR}/data/raw/garmin/garmin.db")

# Check null rates across all key columns in each health table
checks = {
    'hrv': ['calendar_date', 'weekly_avg', 'last_night', 'status', 'baseline_low', 'baseline_upper'],
    'sleep': ['calendar_date', 'sleep_time_seconds', 'deep_sleep_seconds', 'rem_sleep_seconds', 'average_spo2', 'avg_sleep_stress', 'sleep_score_feedback'],
    'body_battery': ['calendar_date', 'charged', 'drained', 'highest', 'lowest', 'at_wake'],
    'stress': ['calendar_date', 'avg_stress', 'max_stress', 'stress_qualifier'],
    'training_readiness': ['calendar_date', 'score', 'level', 'feedback_short', 'recovery_time'],
    'heart_rate': ['calendar_date', 'resting_hr', 'min_hr', 'max_hr', 'avg_hr'],
}

for table, cols in checks.items():
    df = pd.read_sql(f"SELECT {', '.join(cols)} FROM {table} ORDER BY calendar_date DESC", conn)
    total = len(df)
    print(f"\n{table} ({total} rows):")
    for col in cols:
        nulls = df[col].isna().sum()
        if nulls > 0:
            pct = nulls / total * 100
            print(f"  ⚠️  {col}: {nulls} nulls ({pct:.1f}%)")
        else:
            print(f"  ✅ {col}: complete")




hrv (864 rows):
  ✅ calendar_date: complete
  ⚠️  weekly_avg: 6 nulls (0.7%)
  ⚠️  last_night: 864 nulls (100.0%)
  ✅ status: complete
  ⚠️  baseline_low: 18 nulls (2.1%)
  ⚠️  baseline_upper: 18 nulls (2.1%)

sleep (838 rows):
  ✅ calendar_date: complete
  ✅ sleep_time_seconds: complete
  ⚠️  deep_sleep_seconds: 1 nulls (0.1%)
  ⚠️  rem_sleep_seconds: 1 nulls (0.1%)
  ⚠️  average_spo2: 838 nulls (100.0%)
  ⚠️  avg_sleep_stress: 1 nulls (0.1%)
  ⚠️  sleep_score_feedback: 1 nulls (0.1%)

body_battery (3673 rows):
  ✅ calendar_date: complete
  ⚠️  charged: 3673 nulls (100.0%)
  ⚠️  drained: 3673 nulls (100.0%)
  ⚠️  highest: 3673 nulls (100.0%)
  ⚠️  lowest: 3673 nulls (100.0%)
  ⚠️  at_wake: 3673 nulls (100.0%)

stress (3673 rows):
  ✅ calendar_date: complete
  ⚠️  avg_stress: 3673 nulls (100.0%)
  ⚠️  max_stress: 2801 nulls (76.3%)
  ⚠️  stress_qualifier: 3673 nulls (100.0%)

training_readiness (872 rows):
  ✅ calendar_date: complete
  ⚠️  score: 4 nulls (0.5%)
  ✅ level: complete
  ✅

In [9]:
conn = sqlite3.connect(f"{BASE_DIR}/data/raw/garmin/garmin.db")

# Check hrv_timeline for last_night values
print("=== hrv_timeline sample ===")
df_hvt = pd.read_sql("""
    SELECT * FROM hrv_timeline
    ORDER BY calendar_date DESC
    LIMIT 5
""", conn)
print(df_hvt.to_string())

# Check body_battery raw_json for actual values
print("\n=== body_battery raw_json sample ===")
df_bb = pd.read_sql("""
    SELECT calendar_date, raw_json
    FROM body_battery
    ORDER BY calendar_date DESC
    LIMIT 3
""", conn)
for _, row in df_bb.iterrows():
    print(f"\n{row['calendar_date']}:")
    print(row['raw_json'][:500] if row['raw_json'] else "NULL")

# Check stress raw_json
print("\n=== stress raw_json sample ===")
df_st = pd.read_sql("""
    SELECT calendar_date, raw_json
    FROM stress
    ORDER BY calendar_date DESC
    LIMIT 2
""", conn)
for _, row in df_st.iterrows():
    print(f"\n{row['calendar_date']}:")
    print(row['raw_json'][:500] if row['raw_json'] else "NULL")



=== hrv_timeline sample ===
  calendar_date  reading_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [10]:
# ── Step C: Normal authentication (run every session) ─────────────────────────
from stravalib import Client as StravaClient

strava = StravaClient()
token_response = strava.refresh_access_token(
    client_id=STRAVA_CLIENT_ID,
    client_secret=STRAVA_CLIENT_SECRET,
    refresh_token=STRAVA_REFRESH_TOKEN
)
strava.access_token = token_response['access_token']

athlete = strava.get_athlete()
print(f"✅ Strava authenticated as: {athlete.firstname} {athlete.lastname}")

  return datetime.utcnow().replace(tzinfo=utc)



✅ Strava authenticated as: Will Sutherland


In [11]:
# ── Pull historical runs ───────────────────────────────────────────────────────
import json
from datetime import datetime, timedelta, timezone

HISTORY_DAYS = 180
after_dt = datetime.now(timezone.utc) - timedelta(days=HISTORY_DAYS)

print(f"Pulling runs from the last {HISTORY_DAYS} days...")

all_activities = list(strava.get_activities(after=after_dt, limit=200))
runs = [a for a in all_activities if 'Run' in str(a.type)]

print(f"✅ Retrieved {len(runs)} runs from Strava.")

strava_raw = [
    {
        'strava_id':          str(a.id),
        'name':               a.name,
        'start_date':         str(a.start_date_local),
        'distance_m':         float(a.distance or 0),
        'moving_time_s':      int(a.moving_time) if a.moving_time else 0,
        'elapsed_time_s':     int(a.elapsed_time) if a.elapsed_time else 0,
        'total_elevation_m':  float(a.total_elevation_gain or 0),
        'average_heartrate':  a.average_heartrate,
        'max_heartrate':      a.max_heartrate,
        'average_speed_ms':   float(a.average_speed or 0),
        'suffer_score':       a.suffer_score,
        'workout_type':       a.workout_type,
    }
    for a in runs
]

strava_raw_path = os.path.join(STRAVA_RAW_DIR, "activities_raw.json")
with open(strava_raw_path, 'w') as f:
    json.dump(strava_raw, f, indent=2, default=str)

print(f"💾 Saved to {strava_raw_path}")
if strava_raw:
    preview_cols = ['name', 'start_date', 'distance_m', 'moving_time_s', 'average_heartrate', 'suffer_score']
    display(pd.DataFrame(strava_raw[:5])[preview_cols])

Pulling runs from the last 180 days...


✅ Retrieved 127 runs from Strava.


  return datetime.utcnow().replace(tzinfo=utc)



💾 Saved to /content/drive/MyDrive/running_coach/data/raw/strava/activities_raw.json


,name,start_date,distance_m,moving_time_s,average_heartrate,suffer_score
0,Lunch Run,2025-11-22 11:26:10+00:00,10409.8,3208,144.3,47
1,Afternoon Run,2025-11-25 16:49:39+00:00,8114.6,2527,132.7,16
2,Morning Run,2025-11-27 06:09:13+00:00,8255.0,2401,147.8,41
3,Morning Run,2025-11-29 09:18:23+00:00,14412.0,4404,133.3,27
4,Morning Run,2025-11-30 08:03:16+00:00,10183.9,2938,141.9,34


---
## 7. Normalize Strava to Shared WorkoutRecord Schema

All activities are normalized to a consistent schema before touching any agent or tool.
Garmin-only fields default to `None` here — they are backfilled from `garmin.db` in Section 8.

```
WorkoutRecord
├── activity_id        str    — 'strava_12345'
├── source             str    — 'strava'
├── date               str    — YYYY-MM-DD
├── name               str    — Activity name
├── distance_km        float
├── duration_min       float
├── avg_pace_min_km    float
├── avg_hr             int
├── max_hr             int
├── elevation_m        float
├── suffer_score       int    — Strava only
├── training_load      float  — Garmin only (backfilled in Section 8)
├── aerobic_effect     float  — Garmin only (backfilled in Section 8)
├── anaerobic_effect   float  — Garmin only (backfilled in Section 8)
└── garmin_enriched    bool
```

In [12]:
def normalize_strava(activity: dict) -> dict:
    """Normalize a raw Strava activity dict to the shared WorkoutRecord schema."""
    distance_km  = round(activity.get('distance_m', 0) / 1000, 2)
    duration_min = round(activity.get('moving_time_s', 0) / 60, 2)
    avg_pace     = round(duration_min / distance_km, 2) if distance_km > 0 else None
    date_str     = activity.get('start_date', '')[:10]

    return {
        'activity_id':      f"strava_{activity.get('strava_id', '')}",
        'source':           'strava',
        'date':             date_str,
        'name':             activity.get('name', ''),
        'distance_km':      distance_km,
        'duration_min':     duration_min,
        'avg_pace_min_km':  avg_pace,
        'avg_hr':           activity.get('average_heartrate'),
        'max_hr':           activity.get('max_heartrate'),
        'elevation_m':      activity.get('total_elevation_m'),
        'suffer_score':     activity.get('suffer_score'),
        'training_load':    None,
        'aerobic_effect':   None,
        'anaerobic_effect': None,
        'garmin_enriched':  False,
    }

workouts = [normalize_strava(a) for a in strava_raw]
print(f"✅ {len(workouts)} Strava records normalized.")

✅ 127 Strava records normalized.


---
## 8. Enrich WorkoutRecords from Garmin DB

For each Strava activity, we look up the matching date in `garmin.db` and backfill
Garmin-only fields: training load, aerobic/anaerobic training effect, plus a daily
context snapshot (HRV, sleep score, body battery, training readiness) that the
Recovery Agent will use to make recovery decisions.

The `garmin_enriched` flag is set to `True` on any record where Garmin data was found.

In [13]:
# ── Pull Garmin activity-level fields ─────────────────────────────────────────
# Keyed by date (YYYY-MM-DD) for merge with Strava records
try:
    df_garmin_acts = pd.read_sql("""
        SELECT
            DATE(start_time_local)        AS date,
            start_time_local              AS start_time,
            training_load                 AS training_load,
            activity_name                 AS garmin_name,
            aerobic_training_effect       AS aerobic_effect,
            anaerobic_training_effect     AS anaerobic_effect
        FROM activity
        WHERE LOWER(activity_type) LIKE '%run%'
        ORDER BY start_time_local DESC
    """, conn)
    df_garmin_acts = df_garmin_acts.set_index('date')
    # Store as list per date to handle multiple runs on same day
    garmin_act_by_date = {}
    for _, row in df_garmin_acts.reset_index().iterrows():
        date = row['date']
        if date not in garmin_act_by_date:
            garmin_act_by_date[date] = []
        garmin_act_by_date[date].append(row.to_dict())
    print(f"✅ Garmin activity records loaded: {len(garmin_act_by_date)} dates")
except Exception as e:
    garmin_act_by_date = {}
    print(f"⚠️  Could not load Garmin activities: {e}")

# ── Pull daily health context ──────────────────────────────────────────────────
# HRV, sleep, body battery, training readiness — used by Recovery Agent
health_queries = {
    'hrv':                "SELECT calendar_date, weekly_avg, last_night, status, baseline_low, baseline_upper FROM hrv",
    'sleep':              "SELECT calendar_date, sleep_time_seconds, deep_sleep_seconds, rem_sleep_seconds, average_spo2, avg_sleep_stress, sleep_score_feedback FROM sleep",
    'body_battery':       "SELECT calendar_date, charged, drained, highest, lowest, at_wake FROM body_battery",
    'training_readiness': "SELECT calendar_date, score, level, feedback_short, recovery_time, hrv_factor_feedback, sleep_history_factor_feedback FROM training_readiness",
    'heart_rate':         "SELECT calendar_date, resting_hr FROM heart_rate",
    'stress':             "SELECT calendar_date, avg_stress, max_stress, stress_qualifier FROM stress",
}

health_by_date = {}  # date -> {hrv_status, sleep_score, body_battery, ...}

for table, query in health_queries.items():
    try:
        df = pd.read_sql(query, conn)
        df = df.set_index('calendar_date')
        for date, row in df.iterrows():
            if date not in health_by_date:
                health_by_date[date] = {}
            for col, val in row.items():
                health_by_date[date][f"{table}_{col}"] = val
        print(f"  ✅ {table}: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️  {table}: {e}")

print(f"\n✅ Daily health context loaded for {len(health_by_date)} dates.")

  return datetime.utcnow().replace(tzinfo=utc)



✅ Garmin activity records loaded: 550 dates
  ✅ hrv: 864 records
  ✅ sleep: 838 records
  ✅ body_battery: 3673 records
  ✅ training_readiness: 872 records
  ✅ heart_rate: 3673 records
  ✅ stress: 3673 records

✅ Daily health context loaded for 3673 dates.


In [14]:
# ── Merge Garmin enrichment into WorkoutRecords ────────────────────────────────
from datetime import datetime

GARMIN_ACT_FIELDS = ['training_load', 'aerobic_effect', 'anaerobic_effect', 'garmin_name']
enriched_count = 0

def closest_garmin_activity(strava_start: str, garmin_activities: list) -> dict | None:
    """
    Given a Strava start datetime string and a list of Garmin activities on the same
    date, return the Garmin activity with the closest start time.
    """
    if not garmin_activities:
        return None
    if len(garmin_activities) == 1:
        return garmin_activities[0]
    try:
        strava_dt = datetime.fromisoformat(strava_start)
        def time_diff(g):
            try:
                return abs((datetime.fromisoformat(g['start_time']) - strava_dt).total_seconds())
            except:
                return float('inf')
        return min(garmin_activities, key=time_diff)
    except:
        return garmin_activities[0]

for record in workouts:
    date = record.get('date')
    if not date:
        continue

    # Match to closest Garmin activity on the same date by start time
    if date in garmin_act_by_date:
        strava_start = record.get('start_date', '')
        garmin_act   = closest_garmin_activity(strava_start, garmin_act_by_date[date])
        if garmin_act:
            for field in GARMIN_ACT_FIELDS:
                if garmin_act.get(field) is not None:
                    record[field] = garmin_act[field]
            record['garmin_enriched'] = True
            enriched_count += 1

    # Add daily health context snapshot
    record['health_context'] = health_by_date.get(date, {})

# Sort most recent first
all_workouts = sorted(workouts, key=lambda r: r.get('date') or '', reverse=True)

# Save
processed_path = os.path.join(PROC_DATA_DIR, "workouts_normalized.json")
with open(processed_path, 'w') as f:
    json.dump(all_workouts, f, indent=2, default=str)

print(f"✅ {len(all_workouts)} WorkoutRecords saved to {processed_path}")
print(f"   Garmin activity-enriched: {enriched_count} / {len(all_workouts)}")
print(f"   Health context attached:  {sum(1 for w in all_workouts if w.get('health_context'))} / {len(all_workouts)}")

# Preview most recent record in full
print("\nMost recent WorkoutRecord:")
import pprint
pprint.pprint(all_workouts[0])

✅ 127 WorkoutRecords saved to /content/drive/MyDrive/running_coach/data/processed/workouts_normalized.json
   Garmin activity-enriched: 124 / 127
   Health context attached:  125 / 127

Most recent WorkoutRecord:
{'activity_id': 'strava_18579334719',
 'aerobic_effect': None,
 'anaerobic_effect': None,
 'avg_hr': 121.3,
 'avg_pace_min_km': 5.13,
 'date': '2026-05-20',
 'distance_km': 8.03,
 'duration_min': 41.22,
 'elevation_m': 147.0,
 'garmin_enriched': False,
 'health_context': {},
 'max_hr': 141,
 'name': 'Morning Run',
 'source': 'strava',
 'suffer_score': 8,
 'training_load': None}


---
## 9. Smoke Tests — Verify Everything is Wired Up

In [15]:
# ── Test 1: Gemini API ─────────────────────────────────────────────────────────
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(
    "You are a running coach. In one sentence, what is the most common mistake "
    "half-marathon runners make in their training?"
)
print("Gemini smoke test:")
print(response.text)

  return datetime.utcnow().replace(tzinfo=utc)


All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)

  ioloop.make_current()



Gemini smoke test:
The most common mistake half-marathon runners make in their training is running too many of their runs in the "gray zone," failing to go easy enough on easy days and hard enough on hard days.


In [16]:
# ── Test 2: ChromaDB ───────────────────────────────────────────────────────────
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
test_col = chroma_client.get_or_create_collection("smoke_test")

test_col.add(
    documents=["Easy 10km run at Z2 pace, felt good, no fatigue."],
    ids=["smoke_test_1"]
)
results = test_col.query(query_texts=["easy run recovery"], n_results=1)
print("ChromaDB smoke test:")
print(f"  Top result: {results['documents'][0][0]}")

chroma_client.delete_collection("smoke_test")
print("✅ ChromaDB working correctly.")

  return datetime.utcnow().replace(tzinfo=utc)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 46.1MiB/s]


ChromaDB smoke test:
  Top result: Easy 10km run at Z2 pace, felt good, no fatigue.
✅ ChromaDB working correctly.


In [17]:
# ── Test 3: Garmin DB query ────────────────────────────────────────────────────
latest_hrv = pd.read_sql("""
    SELECT calendar_date, weekly_avg, last_night, status
    FROM hrv
    ORDER BY calendar_date DESC
    LIMIT 1
""", conn)
print("Garmin DB smoke test — latest HRV record:")
display(latest_hrv)
print("✅ Garmin DB queryable from Colab.")

  return datetime.utcnow().replace(tzinfo=utc)



Garmin DB smoke test — latest HRV record:


,calendar_date,weekly_avg,last_night,status
0,2026-05-18,82.0,None,LOW


✅ Garmin DB queryable from Colab.


In [18]:
# ── Test 4: WorkoutRecord schema & summary stats ───────────────────────────────
REQUIRED_FIELDS = [
    'activity_id', 'source', 'date', 'name', 'distance_km',
    'duration_min', 'avg_pace_min_km', 'avg_hr', 'max_hr',
    'elevation_m', 'suffer_score', 'training_load',
    'aerobic_effect', 'anaerobic_effect', 'garmin_enriched', 'health_context'
]

schema_ok = True
for i, w in enumerate(all_workouts[:20]):
    missing = [f for f in REQUIRED_FIELDS if f not in w]
    if missing:
        print(f"⚠️  Record {i} ({w.get('date')}) missing: {missing}")
        schema_ok = False

if schema_ok:
    print("✅ Schema validation passed.")

df = pd.DataFrame(all_workouts)
print(f"\nWorkout summary (last {HISTORY_DAYS} days):")
print(f"  Total runs:                  {len(df)}")
print(f"  Total distance (km):         {df['distance_km'].sum():.1f}")
print(f"  Avg distance (km):           {df['distance_km'].mean():.1f}")
print(f"  Avg pace (min/km):           {df['avg_pace_min_km'].mean():.2f}")
print(f"  Avg HR (bpm):                {df['avg_hr'].dropna().mean():.0f}")
print(f"  Garmin activity-enriched:    {df['garmin_enriched'].sum()} / {len(df)}")
print(f"  Health context attached:     {df['health_context'].apply(bool).sum()} / {len(df)}")

conn.close()
print(f"\n✅ Phase 1 complete. Ready for Phase 2 (Tool Layer).")
conn.close()
print("✅ Garmin DB connection closed.")

✅ Schema validation passed.

Workout summary (last 180 days):
  Total runs:                  127
  Total distance (km):         1300.5
  Avg distance (km):           10.2
  Avg pace (min/km):           4.90
  Avg HR (bpm):                139
  Garmin activity-enriched:    124 / 127
  Health context attached:     125 / 127

✅ Phase 1 complete. Ready for Phase 2 (Tool Layer).
✅ Garmin DB connection closed.


---
## ✅ Phase 1 Complete

**What's now in place:**
- All dependencies installed
- Credentials loaded from Colab Secrets (Strava only — no Garmin login in Colab)
- Google Drive mounted, folder structure initialized
- `config.py` written with athlete profile, zone thresholds, and path constants
- `garmin.db` connected and key tables verified (HRV, sleep, body battery, activities, etc.)
- Strava API authenticated, last 90 days of runs pulled
- WorkoutRecords normalized and enriched with Garmin activity data + daily health context
- Gemini API, ChromaDB, and Garmin DB all verified working

**Ongoing data workflow:**

| Action | What to do |
|---|---|
| Refresh Garmin data | Run `garmin-givemydata --days 7` locally → overwrite `garmin.db` in Drive |
| Refresh Strava data | Re-run Section 6 pull cell + Section 8 merge |

**Next — Phase 2: Tool Layer**
- `tools/parse_workout_data.py` — load + validate WorkoutRecords
- `tools/calculate_pace_zones.py` — classify by pace and HR zone
- `tools/calculate_training_load.py` — ATL, CTL, TSB from rolling history
- `tools/query_garmin_db.py` — reusable query functions for HRV, sleep, body battery